# Day 9 — Handling Missing Data and Categorical Features

## Goal
Build a complete machine-learning pipeline that accepts raw customer data, handles missing values, encodes categorical features, and predicts customer churn.

**Target:** `churned`
- `0` = Customer stayed
- `1` = Customer canceled

> Complete one task at a time. Write your own code in the empty code cells before asking for help.

## Step 1 — Create the dataset
Run this starter code. You do not need to change it.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)

n = 240

df = pd.DataFrame({
    "age": np.random.randint(18, 70, n),
    "monthly_spend": np.random.randint(20, 150, n).astype(float),
    "support_tickets": np.random.randint(0, 8, n),
    "plan_type": np.random.choice(["Basic", "Standard", "Premium"], n),
    "auto_pay": np.random.choice(["Yes", "No"], n)
})

risk = (
    -2
    + 0.45 * df["support_tickets"]
    - 0.012 * df["monthly_spend"]
    + 0.8 * (df["plan_type"] == "Basic")
    + 0.7 * (df["auto_pay"] == "No")
)

probability = 1 / (1 + np.exp(-risk))
df["churned"] = np.random.binomial(1, probability)

df.loc[np.random.choice(n, 20, replace=False), "monthly_spend"] = np.nan
df.loc[np.random.choice(n, 15, replace=False), "plan_type"] = np.nan

df.head()

,age,monthly_spend,support_tickets,plan_type,auto_pay,churned
0,56,NaN,0,Basic,Yes,0
1,69,142.0,5,Standard,No,1
2,46,57.0,0,Premium,No,0
3,32,43.0,0,Basic,No,0
4,60,88.0,5,Standard,Yes,0


## Task 1 — Explore the data
Use pandas to answer each question. Do not delete rows containing missing values.

1. How many rows and columns are in the dataset?
2. Which columns contain missing values, and how many?
3. How many customers stayed and how many churned?
4. What percentage of customers churned?
5. Which columns are numerical and which are categorical?

### Your Task 1 code

In [3]:
df.shape

(240, 6)

In [4]:
df.isnull().sum()

age                 0
monthly_spend      20
support_tickets     0
plan_type          15
auto_pay            0
churned             0
dtype: int64

In [5]:
df["churned"].value_counts()

churned
0    164
1     76
Name: count, dtype: int64

In [6]:
df["churned"].value_counts(normalize=True)[1]* 100 # this one shows just the ones that churned. 
df["churned"].value_counts(normalize=True)* 100 


churned
0    68.333333
1    31.666667
Name: proportion, dtype: float64

In [7]:
numerical = df.select_dtypes(include=np.number).columns.tolist()
categorical = df.select_dtypes(include="object").columns.tolist()

print("Numerical columns:", numerical)
print("Categorical columns:", categorical)

Numerical columns: ['age', 'monthly_spend', 'support_tickets', 'churned']
Categorical columns: ['plan_type', 'auto_pay']


### Task 1 explanation
Write what your output tells you in your own words.

**My explanation:** The dataset has 240 customers and 6 columns. Monthly spend has 20 missing values, and plan type has 15. There are 164 customers who stayed and 76 who churned. About 31.67% of the customers churned. The dataset contains numerical and categorical columns, so the missing values and categories will need to be handled before training the model.



## Task 2 — Prepare the features and target
1. Create `X` containing all input features.
2. Create `y` containing the `churned` target.
3. Create `numeric_features` and `categorical_features` lists.
4. Split the data into training and testing sets using:
   - `test_size=0.2`
   - `random_state=42`
   - `stratify=y`

### Your Task 2 code

In [8]:
X = df.drop("churned", axis=1)
y = df["churned"]

X_numerical = X.select_dtypes(include= "number").columns.to_list()
X_categorical = X.select_dtypes(include=object).columns.to_list()

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Task 2 check
After writing your split, check the shapes of `X_train`, `X_test`, `y_train`, and `y_test`.

**Question:** Why are the training sets larger than the testing sets? The trainig set contains 80% of the data and the testing contains 20%. We will use the 80 to train the data whiles testing it on the 20

In [10]:
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training labels shape:", y_train.shape)
print("Test labels shape:", y_test.shape)

Training set shape: (192, 5)
Test set shape: (48, 5)
Training labels shape: (192,)
Test labels shape: (48,)


## Task 3 — Build the preprocessing pipeline
Create two separate pipelines.

### Numerical pipeline
1. Replace missing numerical values with the median.
2. Standardize the numerical features.

### Categorical pipeline
1. Replace missing categorical values with the most frequent category.
2. Convert categories into numbers with `OneHotEncoder`.

Then combine both pipelines using a `ColumnTransformer`.

**Tools to import:** `Pipeline`, `ColumnTransformer`, `SimpleImputer`, `StandardScaler`, and `OneHotEncoder`.

### Your Task 3 code

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NumPipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler())
])

cat_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("onehot", OneHotEncoder(handle_unknown="ignore"))  
])

transformer = ColumnTransformer([
    ("num", NumPipe, X_numerical),
    ("cat", cat_pipe, X_categorical)
])




## Task 4 — Create and train the complete model
1. Import `LogisticRegression`.
2. Create one final pipeline containing your preprocessor and model.
3. Fit the final pipeline using only `X_train` and `y_train`.

**Important:** Do not preprocess the full dataset before splitting it.

### Your Task 4 code

In [12]:
from sklearn.linear_model import LogisticRegression
logreg = Pipeline([
    ("transformer", transformer),
    ("classifier", LogisticRegression())
])



In [13]:
logreg.fit(X_train, y_train)

,steps,"[('transformer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Task 5 — Evaluate the model
Use the trained pipeline to predict `X_test`. Calculate and display:

- Accuracy
- Precision
- Recall
- F1 score
- Confusion matrix

### Your Task 5 code

In [14]:
from sklearn.metrics import (confusion_matrix, classification_report)

y_pred = logreg.predict(X_test)

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.82      0.76        33
           1       0.40      0.27      0.32        15

    accuracy                           0.65        48
   macro avg       0.56      0.54      0.54        48
weighted avg       0.61      0.65      0.62        48

[[27  6]
 [11  4]]


### Explain your results
Answer in your own words.

1. How accurate was your model? The model is not good at finding churned customers (65%)
2. Did it do better at finding customers who stayed or customers who churned? it did better at finding customers who stayed 
3. What does your confusion matrix show? that the model has a harder time finding customers who churned it missed 11 of them

**My explanation:**



## Bonus — Predict a new customer
Create a one-row DataFrame for this customer and use your complete pipeline to predict churn.

| Feature | Value |
| --- | --- |
| age | 25 |
| monthly_spend | Missing |
| support_tickets | 5 |
| plan_type | Premium |
| auto_pay | No |

Do not manually fill the missing monthly spending value. Let your pipeline handle it.

### Your bonus code


In [15]:
new_customer = pd.DataFrame({
    "age": [25],
    "monthly_spend": [np.nan],
    "support_tickets": [5],
    "plan_type": ["Premium"],
    "auto_pay": ["No"]
})

prediction = logreg.predict(new_customer)
print(prediction)


[1]


## Concept questions
Answer these in your own words after finishing the code.

### Question 1
Why might replacing missing numerical values with the median be better than using the mean?

**My answer:** Extream values can pull the mean towards them 

### Question 2
Why can't Logistic Regression directly process categories such as `Premium`?

**My answer:** Becuse its not in numbers form it cant read strings 

### Question 3
What is data leakage, and how does using a pipeline help prevent it?

**My answer:** information that should be unavilable during training influences the model. The pipeline helps when fitting on training data its preprocessing lrans from that data only 

### Question 4
Why should the imputer be fitted using only the training data?

**My answer:** The imputer should learn its filling values only from the training data. Otherwise, information from the test data could influence training and make the evaluation less fair.



## Day 9 reflection
1. What part was most difficult?
    I never tested my model on a new customer or anything before.
2. What mistake did you fix?
I initially passed whole DataFrames to ColumnTransformer instead of lists of column names. I also compared training labels with test predictions, which caused an error because they had different lengths. I fixed this by comparing y_test with y_pred.
3. What did you learn about pipelines?
I learned how to combine preprocessing and a classifier into one pipeline. The pipeline learns preprocessing values from the training data and applies the same steps to test data and new customers. I also learned how to read a confusion matrix and saw that my model was better at identifying customers who stayed than customers who churned.
**My reflection:**

